In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# 1. Đọc dữ liệu thô
df = pd.read_csv('../data/raw/lung_cancer.csv')
print(f"Kích thước bộ dữ liệu ban đầu: {df.shape}")

Kích thước bộ dữ liệu ban đầu: (5000, 30)


In [2]:
# 2. Tách toàn bộ 29 features (X) và target (y)
X = df.drop(columns=['lung_cancer_risk']).copy()
y = df['lung_cancer_risk']

# Phân lập các nhóm biến rõ ràng
continuous_cols = [
    'age', 'education_years', 'income_level', 'smoking_years', 'cigarettes_per_day', 
    'pack_years', 'air_pollution_index', 'bmi', 'oxygen_saturation', 
    'fev1_x10', 'crp_level', 'exercise_hours_per_week', 'diet_quality', 
    'alcohol_units_per_week', 'healthcare_access'
]

categorical_cols = [
    'gender', 'smoker', 'passive_smoking', 'occupational_exposure', 'radon_exposure',
    'family_history_cancer', 'copd', 'asthma', 'previous_tb', 'chronic_cough',
    'chest_pain', 'shortness_of_breath', 'fatigue', 'xray_abnormal'
]

# 3. Xử lý khuyết dữ liệu (Imputation) thông minh
X_imputed = X.copy()

# A. Biến liên tục: Điền Trung vị (Median)
for col in continuous_cols:
    X_imputed[col] = X_imputed[col].fillna(X_imputed[col].median())

# B. Biến phân loại (0/1): Điền Yếu vị (Mode - giá trị xuất hiện nhiều nhất)
for col in categorical_cols:
    X_imputed[col] = X_imputed[col].fillna(X_imputed[col].mode()[0])

# C. Ép kiểu số nguyên cho biến phân loại để khớp 100% với Frontend (thẻ <select>)
for col in categorical_cols:
    X_imputed[col] = X_imputed[col].astype(int)

print("Đã kiểm tra và xử lý xong các giá trị thiếu (Bao gồm Median và Mode)!")

Đã kiểm tra và xử lý xong các giá trị thiếu (Bao gồm Median và Mode)!


In [3]:
# 4. Chia tập dữ liệu (80% Train - 20% Test)
# stratify=y đảm bảo tỉ lệ người mắc bệnh ở tập Train và Test là như nhau
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Kích thước tập Train: {X_train.shape}")
print(f"Kích thước tập Test: {X_test.shape}")

Kích thước tập Train: (4000, 29)
Kích thước tập Test: (1000, 29)


In [4]:
# 5. Xác định các biến liên tục cần chuẩn hóa
scaler = StandardScaler()

# CHỈ FIT trên tập Train để tránh rò rỉ dữ liệu (Data Leakage)
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])

# Dùng bộ scaler đó transform cho tập Test
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("Đã chuẩn hóa (Scale) xong các biến liên tục!")

Đã chuẩn hóa (Scale) xong các biến liên tục!


In [5]:
# 6. Gộp X và y lại và lưu thành file CSV mới
train_processed = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
test_processed = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)

# Lưu vào thư mục processed
train_processed.to_csv('../data/processed/train_processed.csv', index=False)
test_processed.to_csv('../data/processed/test_processed.csv', index=False)

print("Đã xuất file thành công vào thư mục data/processed/!")
print("Sẵn sàng cho bước 03_Modeling_Nature.ipynb")

Đã xuất file thành công vào thư mục data/processed/!
Sẵn sàng cho bước 03_Modeling_Nature.ipynb


In [6]:
import joblib
import os

# Tạo thư mục models nếu chưa có
os.makedirs('../models', exist_ok=True)

# Lưu bộ chuẩn hóa (scaler) ngay tại nơi nó được tạo ra
joblib.dump(scaler, '../models/scaler.pkl')
print("Đã lưu file scaler.pkl thành công!")

Đã lưu file scaler.pkl thành công!


In [7]:
print("// 1. Biến liên tục (Đã lấy Trung vị)")
for col in continuous_cols:
    val = round(X[col].median(), 1) # Làm tròn 1 chữ số
    print(f"{col}: cleanFloat(row.{col} || row.{col.capitalize()}, {val}),")

print("\n// 2. Biến phân loại (Đã lấy Yếu vị)")
for col in categorical_cols:
    val = int(X[col].mode()[0])
    print(f"{col}: cleanInt(row.{col} || row.{col.capitalize()}, {val}),")

// 1. Biến liên tục (Đã lấy Trung vị)
age: cleanFloat(row.age || row.Age, 55.0),
education_years: cleanFloat(row.education_years || row.Education_years, 11.0),
income_level: cleanFloat(row.income_level || row.Income_level, 3.0),
smoking_years: cleanFloat(row.smoking_years || row.Smoking_years, 0.0),
cigarettes_per_day: cleanFloat(row.cigarettes_per_day || row.Cigarettes_per_day, 0.0),
pack_years: cleanFloat(row.pack_years || row.Pack_years, 0.0),
air_pollution_index: cleanFloat(row.air_pollution_index || row.Air_pollution_index, 64.0),
bmi: cleanFloat(row.bmi || row.Bmi, 24.0),
oxygen_saturation: cleanFloat(row.oxygen_saturation || row.Oxygen_saturation, 97.0),
fev1_x10: cleanFloat(row.fev1_x10 || row.Fev1_x10, 33.0),
crp_level: cleanFloat(row.crp_level || row.Crp_level, 3.0),
exercise_hours_per_week: cleanFloat(row.exercise_hours_per_week || row.Exercise_hours_per_week, 2.0),
diet_quality: cleanFloat(row.diet_quality || row.Diet_quality, 3.0),
alcohol_units_per_week: cleanFloat(row.al